# Churn Feature Engineering with PySpark

This notebook builds a Gold feature table for subscription churn prediction from existing Silver clean tables.

**Input tables**

- `silver.users`
- `silver.products`
- `silver.subscription_plans`
- `silver.subscriptions`
- `silver.subscription_changes`
- `silver.payments`
- `silver.license_keys`
- `silver.license_allocations`
- `silver.usage_events`
- `silver.support_tickets`

**Output table**

- `gold.churn_features`

The feature grain is one row per `subscription_id`. The label comes from the existing churn column in `silver.subscriptions`.

## 1. Configuration

For a simple project version, this notebook uses `current_date()` as the feature reference date. For a production-style ML pipeline, use a fixed `snapshot_date` and only include events/payments/tickets/changes that happened before that date.

In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_timestamp, lit, create_map, upper, trim
from pyspark.sql.types import DecimalType
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from functools import reduce
from itertools import chain
import sklearn
print(pyspark.__version__)
print("import successful")

3.5.0
import successful


In [0]:
# !pip install scikit-learn

In [0]:
builder = (
    SparkSession.builder
    .appName("user_bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

In [0]:
silver_db = "/Volumes/datalake_catalog/datalake_schema/silver_draft/"
gold_db = "gold"
output_table = f"{gold_db}.churn_features"
snapshot_date = F.current_date()

## 2. Load Silver Tables

In [0]:
# users = (
#     spark.read
#     .format("delta")
#     .load(silver_db+"/users")
# )

In [0]:
users = (
    spark.read
    .format("delta")
    .load(silver_db+"/users")
)
products = (
    spark.read
    .format("delta")
    .load(silver_db+"/products")
)
plans = (
    spark.read
    .format("delta")
    .load(silver_db+"/plans")
)
subs = (
    spark.read
    .format("delta")
    .load(silver_db+"/subscriptions")
).drop("churn")
changes = (
    spark.read
    .format("delta")
    .load(silver_db+"/subscription_changes")
)
payments = (
    spark.read
    .format("delta")
    .load(silver_db+"/payments")
)
licenses = (
    spark.read
    .format("delta")
    .load(silver_db+"/licenses")
)
allocations = (
    spark.read
    .format("delta")
    .load(silver_db+"/license_allocations")
)
usage = (
    spark.read
    .format("delta")
    .load(silver_db+"/usage_events")
)
tickets = (
    spark.read
    .format("delta")
    .load(silver_db+"/support_tickets")
)

In [0]:
usage.columns

['subscription_id',
 'user_id',
 'created_at_event',
 'processed',
 'ingest_time',
 'source_identifier',
 'batch_id',
 'content_id',
 'created_at',
 'device_type',
 'event_id',
 'event_properties_duration_sec',
 'event_properties_referrer',
 'event_timestamp',
 'event_type',
 'feature_name',
 'platform',
 'session_id']

In [0]:
users.show(5)

+-------+----------+----------+--------------------+---------+---+------+-------------------+-------------+-------------------+--------------------+----------------+
|user_id|first_name| last_name|               email|  country|age|gender|acquisition_channel|is_enterprise|         created_at|         ingest_time|is_valid_country|
+-------+----------+----------+--------------------+---------+---+------+-------------------+-------------+-------------------+--------------------+----------------+
|      2|     Linda|    Miller|linda.miller@outl...|    Other| 29|  male|           referral|         true|2023-10-31 00:47:20|2026-04-22 04:25:...|           false|
|     11|    Rajata|Srivastava|rajata.srivastava...|    India| 44|  male|               paid|        false|2024-11-09 22:34:36|2026-04-22 04:25:...|            true|
|     14|      Neel|     Sinha|neel.sinha@gmail.com|    India| 46|female|            organic|         true|2025-12-16 23:21:54|2026-04-22 04:25:...|            true|
|   

## Churn features: subscription churn

For churn prediction in this project, the model should use subscription_id as the main prediction grain. This means each row represents one subscription at a given prediction date, and the target is whether that subscription will churn in a future period (for example, within the next 30 days). user_id can still be kept as a supporting key to join extra customer information such as account age, country, usage behaviour, or support history, but the main entity being predicted is the subscription.

+ Tenure = cutoff_date - subscriptions created_at if end_date is NULL, if end_date is not null and end_date < cuttoff_date, then end_date-subscriptions created_at, if end_date>cutoff_date, then cutoff_date-subscriptions created_at. We dont care whether or not they change subscription in this.
+ usage_frequency = number of usage events in the 90 days before prediction_date (we can use others like 30d, 60d,...)
+ number_of_support_ticket: we need to based on user_id + date range. The created_at in the support_ticket table decides which subsription_id that it belongs to. Note that, one user -> many active subscriptions at the same time is possible.
+ total_spend = sum of all successful payment amounts for that subscription. Because each subscription_id, calculate total_spend as the sum of all successful payment amounts that belong to that subscription, using only payments made on or before the feature cutoff date for churn prediction. Because the subscription_id stays the same even when the plan changes, upgrades or downgrades do not split the spend into different subscriptions—they are all still included in the same subscription’s total_spend.
+ last_interaction: A simple groupBy + max (latest interaction in the usage_event), if no interaction yet, use the created_at of the subscription_id).
+ device_type: count distinct device types or use one-hot encoding device types. It belongs to feature engineering, therefore, at the moment, we use count distnct device types.
+ payment_method: the same method as device_type
+ has_license, max_seats, allocated_seats: A subscription may or may not have a licence, and a licence may or may not have allocation records. The has_license is whether or not the subscription_id has a license, the max_seats is the number of total of max_seats for the license, and allocated_seats are the number of allocated_seats which is active.
+ user_id, age, gender, and country: based on users table.

### Cutoff date => need to fix data gen after this time

the date that separates historical data used for features from future data used for the churn label to avoid date leakage (filter using it later after getting all features)

### Update churn:

churn = Yes if status is cancelled/expired OR end_date <= cutoff_date

In [0]:
cutoff_date = F.lit("2026-04-30").cast("date")

In [0]:
subs = subs.filter(F.col("start_date") <= cutoff_date)
subs.show(1)

+-------+---------------+-------+----------+----------+---------+-----------+-------------------+--------------------+
|user_id|subscription_id|plan_id|start_date|  end_date|   status|current_mrr|         created_at|         ingest_time|
+-------+---------------+-------+----------+----------+---------+-----------+-------------------+--------------------+
|    484|             20|      1|2025-05-03|2025-11-30|cancelled|       1.31|2025-05-03 03:30:46|2026-04-22 04:29:...|
+-------+---------------+-------+----------+----------+---------+-----------+-------------------+--------------------+
only showing top 1 row


In [0]:
# Redefind churn
subs = subs.withColumn(
    "churn",
    F.when(
        (
            F.lower(F.col("status")).isin("cancelled", "canceled", "expired")
        )
        |
        (
            F.col("end_date").isNotNull()
            & (F.to_date(F.col("end_date")) <= cutoff_date)
        ),
        F.lit("Yes")
    ).otherwise(F.lit("No"))
)

In [0]:
subs.count()

205585

In [0]:
subs.groupBy("churn").agg(
    F.count("churn")).show()

+-----+------------+
|churn|count(churn)|
+-----+------------+
|  Yes|       71233|
|   No|      134352|
+-----+------------+



### number_of_support_ticket
support_created_at: Fill NULL => 0 later

In [0]:
df_subs_joined = (
    subs.alias("s")
    .join(
        tickets.withColumnRenamed("created_at", "support_created_at").select("user_id", "support_created_at").alias("t"),
        on=(
            (F.col("s.user_id") == F.col("t.user_id")) &
            (F.col("t.support_created_at") >= F.col("s.start_date")) &
            (
                F.col("t.support_created_at") <= F.coalesce(
                    F.col("s.end_date"),
                    F.to_timestamp(F.lit("2999-12-31 23:59:59"))
                )
            )
        ),
        how="left"
    ).select(
        "s.*",
        F.col("t.support_created_at")
    )
)

df_support_count = (
    df_subs_joined
    .groupBy("subscription_id")
    .agg(
        F.count("support_created_at").alias("support_ticket_count")
    )
)

df1 = (
    df_subs_joined
    .drop("support_created_at")
    .dropDuplicates(["subscription_id"])
    .join(df_support_count, on="subscription_id", how="left")
)


In [0]:
df1.count()

205585

### tenure

tenure = min(end_date, cutoff_date) - subscription_created_at, and if end_date is NULL, use cutoff_date.

In [0]:
df2 = (
    df1.withColumn(
        "tenure",
        F.datediff(
            F.coalesce(
                F.least(F.col("end_date"), cutoff_date),
                cutoff_date
            ),
            F.col("start_date")
        )
    )
)


df2.show(1)

+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+
|subscription_id|user_id|plan_id|start_date|  end_date|   status|current_mrr|         created_at|         ingest_time|churn|support_ticket_count|tenure|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+
|             30|  15660|     11|2025-05-02|2025-07-05|cancelled|      23.62|2025-05-02 10:57:14|2026-04-22 04:29:...|  Yes|                   0|    64|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+
only showing top 1 row


In [0]:
df_tenure_stats = df2.select(
    F.min("tenure").alias("min_tenure"),
    F.max("tenure").alias("max_tenure")
)
df_tenure_stats.show()

+----------+----------+
|min_tenure|max_tenure|
+----------+----------+
|         0|      1215|
+----------+----------+



### usage_frequency

In [0]:
# usage_frequency
usage_frequency = usage.groupBy("subscription_id").agg(F.count("event_id").alias("usage_frequency"))
usage_frequency.count()

16799

In [0]:
df3 = (
    df2.alias("s")
    .join(
        usage_frequency.alias("af"),
        on="subscription_id",
        how="left"
    )
).fillna({"usage_frequency": 0})

In [0]:
# At the moment, there are subscription_id which do not have because the data is missing. So, if I want to predict something => 0 in usage_frequency
df3.select("status").distinct().show()

+---------+
|   status|
+---------+
|cancelled|
|   active|
|  expired|
|    trial|
+---------+



In [0]:
# number_of_support_ticket in tickets
# Based on the created_at, we will know which subscription_id that it belongs to
tickets.show(1)

+-------+---------+--------+--------------------+-------------------+--------------------+
|user_id|ticket_id|category|         description|         created_at|         ingest_time|
+-------+---------+--------+--------------------+-------------------+--------------------+
|  24816|        2| billing|Hi Support, I not...|2024-10-21 16:01:48|2026-04-22 05:30:...|
+-------+---------+--------+--------------------+-------------------+--------------------+
only showing top 1 row


In [0]:
df3.drop("description").show(6, truncate=False)

+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------------+-----+--------------------+------+---------------+
|subscription_id|user_id|plan_id|start_date|end_date  |status   |current_mrr|created_at         |ingest_time               |churn|support_ticket_count|tenure|usage_frequency|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------------+-----+--------------------+------+---------------+
|42             |23395  |15     |2024-05-30|2024-10-28|expired  |42.27      |2024-05-30 08:28:49|2026-04-22 04:29:16.967562|Yes  |0                   |151   |0              |
|169            |7343   |5      |2026-03-04|2026-08-14|cancelled|67.72      |2026-03-04 06:53:12|2026-04-22 04:29:16.967562|Yes  |0                   |57    |0              |
|286            |43113  |3      |2023-06-05|NULL      |active   |370.16     |2023-06-05 16:37:24|2026-04-22 04:29:16.967562|N

### total_spend

In [0]:
total_spend = (
    payments
    .filter(
        (F.col("payment_status") == "success") &
        (F.col("payment_date") <= cutoff_date)
    )
    .groupBy("subscription_id")
    .agg(F.sum("amount").alias("total_spend"))
)

df4 = (
    df3.alias("s")
    .join(
        total_spend.alias("t"),
        on="subscription_id",
        how="left"
    )
    .fillna({"total_spend": 0})
)

df4.show(2)

+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+
|subscription_id|user_id|plan_id|start_date|  end_date|   status|current_mrr|         created_at|         ingest_time|churn|support_ticket_count|tenure|usage_frequency|total_spend|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+
|             60|  14165|      4|2025-03-16|2025-07-01|cancelled|      41.87|2025-03-16 08:31:01|2026-04-22 04:29:...|  Yes|                   0|   107|              0|      41.87|
|            125|   2735|      8|2025-06-20|      NULL|   active|      11.24|2025-06-20 10:54:57|2026-04-22 04:29:...|   No|                   0|   314|              0|      22.48|
+---------------+-------+-------+----------+----------+---------+-----------+------------------

In [0]:
df4.count()

205585

### last_interaction

In [0]:
last_interaction = (
    usage
    .groupBy("subscription_id")
    .agg(
        F.max("event_timestamp").alias("last_interaction")
    )
)

df5 = (
    df4.alias("d")
    .join(
        last_interaction.alias("l"),
        on="subscription_id",
        how="left"
    )
    .withColumn(
        "last_interaction",
        F.coalesce(F.col("last_interaction"), F.col("created_at"))
    )
)
df5.count()

205585

In [0]:
df5.show()

+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+--------------------+
|subscription_id|user_id|plan_id|start_date|  end_date|   status|current_mrr|         created_at|         ingest_time|churn|support_ticket_count|tenure|usage_frequency|total_spend|    last_interaction|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+--------------------+
|            183|  16921|     12|2026-02-14|      NULL|   active|      12.46|2026-02-14 22:40:27|2026-04-22 04:29:...|   No|                   0|    75|              0|      12.46| 2026-02-14 22:40:27|
|            291|  20571|      2|2025-04-25|      NULL|   active|      10.40|2025-04-25 02:45:45|2026-04-22 04:29:...|   No|                   1|   370|              0|      20.80| 2025-04-25 

### device_type
Later, after joining to df5 all.

Just join, but need to consider that one subscription_id can have multiple device_types

In [0]:
device_features = (
    usage
    .groupBy("subscription_id")
    .agg(
        F.countDistinct("device_type").alias("device_type_count")
    )
)

df6 = (
    df5.alias("d")
    .join(
        device_features.alias("f"),
        on="subscription_id",
        how="left"
    )
).fillna({"device_type_count": 0})
df6.count()

205585

### payment_method

In [0]:
payment_features = (
    payments
    .groupBy("subscription_id")
    .agg(
        F.countDistinct("payment_method").alias("payment_method_count")
    )
)

df7 = (
    df6.alias("d")
    .join(payment_features.alias("pm"), on="subscription_id", how="left")
    .fillna({
        "payment_method_count": 0
    })
)

In [0]:
df7.show(2)

+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+-------------------+-----------------+--------------------+
|subscription_id|user_id|plan_id|start_date|  end_date|   status|current_mrr|         created_at|         ingest_time|churn|support_ticket_count|tenure|usage_frequency|total_spend|   last_interaction|device_type_count|payment_method_count|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+-------------------+-----------------+--------------------+
|             70|  28227|      1|2023-11-24|2024-11-14|cancelled|       1.31|2023-11-24 23:04:23|2026-04-22 04:29:...|  Yes|                   1|   356|              0|     189.24|2023-11-24 23:04:23|                0|                   1|
|            313|  46938|      7|2024-10

### number_of_active_seats_per_license

The relationship in license_allocations: allocations → license_id → licenses → subscription_id

That means we can calculate seat-related features from allocations, then join through licenses, and finally join back to subs by subscription_id.

Logic: count active seats per license_id

Having a licence does not guarantee having allocated seats because licenses is the master list of all licences, while license_allocations is only the subset of licences that have allocation activity.

#### has_license

In [0]:
licence_flag = (
    licenses
    .select("subscription_id")
    .dropDuplicates()
    .withColumn("has_licence", F.lit(1))
)

df8 = (
    df7
    .join(licence_flag, on="subscription_id", how="left")
    .fillna({"has_licence": 0})
)

df8.show(5)


+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+-------------------+-----------------+--------------------+-----------+
|subscription_id|user_id|plan_id|start_date|  end_date|   status|current_mrr|         created_at|         ingest_time|churn|support_ticket_count|tenure|usage_frequency|total_spend|   last_interaction|device_type_count|payment_method_count|has_licence|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+-------------------+-----------------+--------------------+-----------+
|             70|  28227|      1|2023-11-24|2024-11-14|cancelled|       1.31|2023-11-24 23:04:23|2026-04-22 04:29:...|  Yes|                   1|   356|              0|     189.24|2023-11-24 23:04:23|                0|                   1|     

In [0]:
max_seats_by_sub = (
    licenses
    .groupBy("subscription_id")
    .agg(
        F.sum("max_seats").alias("max_seats")
    )
)
allocated_by_license = (
    allocations
    .filter(F.col("status") == "active")
    .groupBy("license_id")
    .agg(
        F.countDistinct("seat_number").alias("allocated_seats")
    )
)

allocated_by_sub = (
    licenses
    .select("license_id", "subscription_id")
    .join(allocated_by_license, on="license_id", how="left")
    .groupBy("subscription_id")
    .agg(
        F.sum("allocated_seats").alias("allocated_seats")
    )
)

df9 = (
    df8
    .join(max_seats_by_sub, on="subscription_id", how="left")
    .join(allocated_by_sub, on="subscription_id", how="left")
    .fillna({
        "max_seats": 0,
        "allocated_seats": 0
    })
)


In [0]:
df9.drop("end_date", "current_mrr").filter((F.col("has_licence")==1) & (F.col("allocated_seats")>0)).show(10)

+---------------+-------+-------+----------+---------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+-------------------+-----------------+--------------------+-----------+---------+---------------+
|subscription_id|user_id|plan_id|start_date|   status|         created_at|         ingest_time|churn|support_ticket_count|tenure|usage_frequency|total_spend|   last_interaction|device_type_count|payment_method_count|has_licence|max_seats|allocated_seats|
+---------------+-------+-------+----------+---------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+-------------------+-----------------+--------------------+-----------+---------+---------------+
|           7443|  17129|      7|2024-02-21|    trial|2024-02-21 07:02:50|2026-04-22 04:29:...|   No|                   0|   799|              0|    2832.65|2024-02-21 07:02:50|                0|                   1|          1|       

In [0]:
df10 = (
    df9.alias("d")
    .join(
        users.select("user_id", "age", "gender", "country").alias("u"),
        on="user_id",
        how="left"
    )
)
df10.count()

205585

In [0]:
avg_usage_time_per_day = (
    usage
    .filter(
        (F.col("event_properties_duration_sec").isNotNull()) &
        (F.col("event_timestamp") <= cutoff_date)
    )
    .withColumn("event_date", F.to_date("event_timestamp"))
    .groupBy("subscription_id", "event_date")
    .agg(
        F.sum("event_properties_duration_sec").alias("daily_usage_time_sec")
    )
    .groupBy("subscription_id")
    .agg(
        F.avg("daily_usage_time_sec").alias("avg_usage_time_per_day_sec")
    )
)


df11 = (
    df10
    .join(avg_usage_time_per_day, on="subscription_id", how="left")
    .fillna({"avg_usage_time_per_day_sec": 0})
)
df11.count()


205585

In [0]:
df11.select(
    F.min("avg_usage_time_per_day_sec").alias("min_tenure"),
    F.max("avg_usage_time_per_day_sec").alias("max_tenure")
).show()


+----------+----------+
|min_tenure|max_tenure|
+----------+----------+
|       0.0|    4422.0|
+----------+----------+



In [0]:
df11.columns

['subscription_id',
 'user_id',
 'plan_id',
 'start_date',
 'end_date',
 'status',
 'current_mrr',
 'created_at',
 'ingest_time',
 'churn',
 'support_ticket_count',
 'tenure',
 'usage_frequency',
 'total_spend',
 'last_interaction',
 'device_type_count',
 'payment_method_count',
 'has_licence',
 'max_seats',
 'allocated_seats',
 'age',
 'gender',
 'country',
 'avg_usage_time_per_day_sec']

## Features:

tenure, usage_frequency, support_ticket_count, total_spend, last_interaction, age, gender, country, payment_method_count, has_licence, max_seats, allocated_seats, avg_usage_time_per_day_sec, subscription_type.

In [0]:
df11.show(1)

+---------------+-------+-------+----------+--------+------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+--------------------+-----------------+--------------------+-----------+---------+---------------+---+------+-------+--------------------------+
|subscription_id|user_id|plan_id|start_date|end_date|status|current_mrr|         created_at|         ingest_time|churn|support_ticket_count|tenure|usage_frequency|total_spend|    last_interaction|device_type_count|payment_method_count|has_licence|max_seats|allocated_seats|age|gender|country|avg_usage_time_per_day_sec|
+---------------+-------+-------+----------+--------+------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+--------------------+-----------------+--------------------+-----------+---------+---------------+---+------+-------+--------------------------+
|             93|  40995|     12|2024-03

In [0]:
df12 = (
    df11.alias("d")
    .join(
        plans.select(
            "plan_id",
            F.col("tier").alias("subscription_type")
        ).alias("p"),
        on="plan_id",
        how="left"
    )
)
df12.show(2)

+-------+---------------+-------+----------+--------+------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+--------------------+-----------------+--------------------+-----------+---------+---------------+---+------+-------+--------------------------+-----------------+
|plan_id|subscription_id|user_id|start_date|end_date|status|current_mrr|         created_at|         ingest_time|churn|support_ticket_count|tenure|usage_frequency|total_spend|    last_interaction|device_type_count|payment_method_count|has_licence|max_seats|allocated_seats|age|gender|country|avg_usage_time_per_day_sec|subscription_type|
+-------+---------------+-------+----------+--------+------+-----------+-------------------+--------------------+-----+--------------------+------+---------------+-----------+--------------------+-----------------+--------------------+-----------+---------+---------------+---+------+-------+--------------------------+-----

In [0]:
df12.count()

205585

In [0]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [0]:
# 1) Select only needed columns from Spark DF -> pandas
pdf = df12.select(
    "created_at",
    "tenure",
    "usage_frequency",
    "support_ticket_count",
    "total_spend",
    "last_interaction",
    "age",
    "gender",
    "country",
    "payment_method_count",
    "has_licence",
    "max_seats",
    "allocated_seats",
    "avg_usage_time_per_day_sec",
    "subscription_type",
    "churn"
).toPandas()



In [0]:
df = pdf.copy()

# 2) Convert label
# If churn is already 0/1, replace this block with: df["churn"] = df["churn"].astype(int)
df["churn"] = df["churn"].map({
    "Yes": 1,
    "No": 0
})
# 3) Convert last_interaction -> numeric feature
df["last_interaction"] = pd.to_datetime(df["last_interaction"], errors="coerce")
cutoff_date_pd = pd.Timestamp("2026-04-30")

df["days_since_last_interaction"] = (
    cutoff_date_pd - df["last_interaction"]
).dt.days

In [0]:
df.head(4)

,created_at,tenure,usage_frequency,support_ticket_count,total_spend,last_interaction,age,gender,country,payment_method_count,has_licence,max_seats,allocated_seats,avg_usage_time_per_day_sec,subscription_type,churn,days_since_last_interaction
0,2024-05-19 02:00:52,711,0,1,41.87,2024-05-19 02:00:52,19,male,India,1,0,0,0,0.0,Premium,0,710
1,2024-04-01 19:29:44,84,0,0,10.40,2024-04-01 19:29:44,35,male,Other,2,0,0,0,0.0,Basic,1,758
2,2024-10-15 02:27:35,562,0,2,41.87,2024-10-15 02:27:35,45,female,India,1,0,0,0,0.0,Premium,0,561
3,2025-12-28 08:51:24,123,0,0,67.72,2025-12-28 08:51:24,42,male,Australia,1,0,0,0,0.0,Premium,0,122


In [0]:
# Drop raw datetime column
df = df.drop(columns=["last_interaction"])

# 4) No imputation -> just drop rows with nulls in any used feature/label
feature_cols = [
    "created_at",
    "tenure",
    "usage_frequency",
    "support_ticket_count",
    "total_spend",
    "days_since_last_interaction",
    "age",
    "gender",
    "country",
    "payment_method_count",
    "has_licence",
    "max_seats",
    "allocated_seats",
    "avg_usage_time_per_day_sec",
    "subscription_type"
]

df = df.dropna(subset=feature_cols + ["churn"]).copy()
df["churn"] = df["churn"].astype(int)

X = df[feature_cols]
y = df["churn"]

In [0]:
# 5) Numeric / categorical columns
numeric_features = [
    "tenure",
    "usage_frequency",
    "support_ticket_count",
    "total_spend",
    "days_since_last_interaction",
    "age",
    "payment_method_count",
    "has_licence",
    "max_seats",
    "allocated_seats",
    "avg_usage_time_per_day_sec"
]

categorical_features = [
    "gender",
    "country",
    "subscription_type"
]

# 6) Preprocessing
# No imputer here
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# 7) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=2001,
    stratify=y
)

In [0]:
X_train.head(5)

,created_at,tenure,usage_frequency,support_ticket_count,total_spend,days_since_last_interaction,age,gender,country,payment_method_count,has_licence,max_seats,allocated_seats,avg_usage_time_per_day_sec,subscription_type
99491,2024-07-12 21:05:09,166,0,0,0.00,656,34,male,Other,0,0,0,0,0.0,Basic
110554,2025-01-20 05:03:48,465,0,0,377.76,464,28,female,Germany,3,0,0,0,0.0,Basic
128500,2023-03-02 19:37:59,1155,0,0,81.24,1154,22,male,Australia,3,0,0,0,0.0,Standard
56450,2024-02-19 16:51:10,267,0,0,0.00,800,36,male,Germany,0,0,0,0,0.0,Premium
95946,2024-02-20 14:10:03,800,0,1,12.20,799,34,male,Spain,1,0,0,0,0.0,Basic


In [0]:
X_test.head(5)

,created_at,tenure,usage_frequency,support_ticket_count,total_spend,days_since_last_interaction,age,gender,country,payment_method_count,has_licence,max_seats,allocated_seats,avg_usage_time_per_day_sec,subscription_type
97814,2024-02-08 21:08:10,812,0,0,0.00,811,44,female,Other,0,0,0,0,0.0,Basic
46908,2024-03-25 17:02:46,766,0,1,23.62,765,52,female,Other,1,0,0,0,0.0,Standard
22571,2023-12-04 02:56:56,878,0,0,10.40,877,38,male,Singapore,1,0,0,0,0.0,Basic
117450,2024-10-25 22:37:39,552,0,0,70.86,551,18,male,Germany,3,0,0,0,0.0,Standard
156239,2026-02-23 01:39:05,66,0,0,41.87,65,18,female,Germany,1,0,0,0,0.0,Premium


In [0]:
# train_df = df[df["created_at"] < cutoff_date_pd].copy()
# test_df  = df[df["created_at"] >= cutoff_date_pd].copy()

In [0]:
# X_train = train_df[feature_cols]
# y_train = train_df["churn"]

# X_test = test_df[feature_cols]
# y_test = test_df["churn"]

In [0]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((143909, 15), (61676, 15), (143909,), (61676,))

In [0]:
y_test.value_counts()

0    40306
1    21370
Name: churn, dtype: int64

In [0]:
# 8) Models
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=2001
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=2001,
        n_jobs=-1
    )
    # "GradientBoosting": GradientBoostingClassifier(
    #     random_state=2001
    # )
    # "SVC": SVC(
    #     class_weight="balanced",
    #     random_state=2001
    # )
}

# 9) Train + evaluate
results = []

for model_name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    results.append({
        "model": model_name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1
    })

    print(f"\n{'='*60}")
    print(f"MODEL: {model_name}")
    print(f"{'='*60}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

# 10) Compare models
results_df = pd.DataFrame(results).sort_values(
    by=["f1_score", "recall", "precision", "accuracy"],
    ascending=False
)

print("\nModel comparison:")
print(results_df)


MODEL: LogisticRegression
Accuracy : 0.9348
Precision: 0.8963
Recall   : 0.9179
F1 Score : 0.9070

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95     40306
           1       0.90      0.92      0.91     21370

    accuracy                           0.93     61676
   macro avg       0.93      0.93      0.93     61676
weighted avg       0.94      0.93      0.93     61676


MODEL: RandomForest
Accuracy : 0.9387
Precision: 0.9490
Recall   : 0.8699
F1 Score : 0.9077

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.98      0.95     40306
           1       0.95      0.87      0.91     21370

    accuracy                           0.94     61676
   macro avg       0.94      0.92      0.93     61676
weighted avg       0.94      0.94      0.94     61676


Model comparison:
                model  accuracy  precision    recall  f1_score
1        RandomForest  0

In [0]:
# 8) Models
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=2001
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=2001,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=2001
    )
    # "SVC": SVC(
    #     class_weight="balanced",
    #     random_state=2001
    # )
}

# 9) Train + evaluate
results = []

for model_name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf.fit(X_train, y_train)

    # Predict on train and test
    y_train_pred = clf.predict(X_train)
    y_test_pred = clf.predict(X_test)

    # Train metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    train_prec = precision_score(y_train, y_train_pred, zero_division=0)
    train_rec = recall_score(y_train, y_train_pred, zero_division=0)
    train_f1 = f1_score(y_train, y_train_pred, zero_division=0)

    # Test metrics
    test_acc = accuracy_score(y_test, y_test_pred)
    test_prec = precision_score(y_test, y_test_pred, zero_division=0)
    test_rec = recall_score(y_test, y_test_pred, zero_division=0)
    test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

    results.append({
        "model": model_name,
        "train_accuracy": train_acc,
        "train_precision": train_prec,
        "train_recall": train_rec,
        "train_f1_score": train_f1,
        "test_accuracy": test_acc,
        "test_precision": test_prec,
        "test_recall": test_rec,
        "test_f1_score": test_f1
    })

    print(f"\n{'='*60}")
    print(f"MODEL: {model_name}")
    print(f"{'='*60}")

    print("\nTRAIN METRICS")
    print(f"Accuracy : {train_acc:.4f}")
    print(f"Precision: {train_prec:.4f}")
    print(f"Recall   : {train_rec:.4f}")
    print(f"F1 Score : {train_f1:.4f}")
    print("\nTrain Classification Report:")
    print(classification_report(y_train, y_train_pred, zero_division=0))

    print("\nTEST METRICS")
    print(f"Accuracy : {test_acc:.4f}")
    print(f"Precision: {test_prec:.4f}")
    print(f"Recall   : {test_rec:.4f}")
    print(f"F1 Score : {test_f1:.4f}")
    print("\nTest Classification Report:")
    print(classification_report(y_test, y_test_pred, zero_division=0))

# 10) Compare models
results_df = pd.DataFrame(results).sort_values(
    by=["test_f1_score", "test_recall", "test_precision", "test_accuracy"],
    ascending=False
)

print("\nModel comparison:")
print(results_df)



MODEL: LogisticRegression

TRAIN METRICS
Accuracy : 0.9369
Precision: 0.9004
Recall   : 0.9197
F1 Score : 0.9100

Train Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.95      0.95     94046
           1       0.90      0.92      0.91     49863

    accuracy                           0.94    143909
   macro avg       0.93      0.93      0.93    143909
weighted avg       0.94      0.94      0.94    143909


TEST METRICS
Accuracy : 0.9348
Precision: 0.8963
Recall   : 0.9179
F1 Score : 0.9070

Test Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95     40306
           1       0.90      0.92      0.91     21370

    accuracy                           0.93     61676
   macro avg       0.93      0.93      0.93     61676
weighted avg       0.94      0.93      0.93     61676


MODEL: RandomForest

TRAIN METRICS
Accuracy : 0.9999
Precision: 0.9998
Recall   : 0.99

In [0]:
spark.stop()